# Explainable and Adaptive Machine Learning for Fraud Risk Detection

**Clean reproducibility notebook for the BIBM fraud-risk project**

This notebook is a cleaned, top-to-bottom version of the original research notebook
`BIBM_Fraud_Detection_Final.ipynb`. Recovery/debug cells, repeated Drive mounts,
hard-coded personal Drive paths, saved-model reloads, and backup-archive cells were removed.

The **research methodology is preserved**:

- PaySim synthetic mobile-money benchmark
- CASH_OUT + TRANSFER subset
- chronological split: train steps 1–520, validation 521–632, test 633–743
- leakage-conscious historical features constructed from preceding observations
- Logistic Regression, Random Forest, Clean XGBoost, and LightGBM
- validation-only threshold selection
- Platt calibration on validation predictions
- SHAP global/local explanations
- adaptive top-1% risk selection inside 24-step windows

> **Data note:** PaySim is synthetic and is not Bangladesh banking/MFS transaction data.
> The dataset CSV is intentionally not included in the repository.

## How to run

1. Obtain the PaySim CSV (`PS_20174392719_1491204439457_log.csv`).
2. Either place it at `data/PS_20174392719_1491204439457_log.csv`, or set the environment variable `PAYSIM_CSV` to its path.
3. Run the notebook from top to bottom.

Outputs are written to relative repository folders:

- `results/`
- `figures/`
- `config/`

The original notebook contained an exploratory **weighted XGBoost** branch. It is omitted here because the accepted paper's final comparison reports four models: Logistic Regression, Random Forest, Clean XGBoost, and LightGBM.

In [ ]:
# If needed in a fresh Colab/Jupyter environment, uncomment:
# %pip install -q pandas numpy scikit-learn xgboost lightgbm shap matplotlib joblib openpyxl

In [ ]:
from pathlib import Path
import os
import json
import sys
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42

DATA_PATH = Path(
    os.environ.get(
        "PAYSIM_CSV",
        "data/PS_20174392719_1491204439457_log.csv",
    )
)

RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
CONFIG_DIR = Path("config")

for directory in (RESULTS_DIR, FIGURES_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = [
    "step",
    "hour",
    "day",
    "is_night",
    "amount",
    "type_transfer",
    "origin_prev_count",
    "dest_prev_count",
    "origin_prev_avg_amount",
    "amount_to_origin_avg",
]

print("Dataset path:", DATA_PATH)
print("Output folders ready.")

## 1. Load PaySim and define the fraud-relevant subset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"PaySim CSV not found at: {DATA_PATH}\n"
        "Place the CSV in data/ or set the PAYSIM_CSV environment variable."
    )

df = pd.read_csv(DATA_PATH)

required_columns = {
    "step", "type", "amount", "nameOrig", "nameDest", "isFraud"
}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Missing required PaySim columns: {sorted(missing)}")

print("Full dataset shape:", df.shape)
print("Fraud cases:", int(df["isFraud"].sum()))
print("Fraud rate:", float(df["isFraud"].mean()))

df_sub = df[df["type"].isin(["CASH_OUT", "TRANSFER"])].copy()

print("\nCASH_OUT + TRANSFER subset:", df_sub.shape)
print("Subset fraud cases:", int(df_sub["isFraud"].sum()))
print("Subset fraud rate:", float(df_sub["isFraud"].mean()))

## 2. Chronological split and leakage-conscious feature engineering

In [ ]:
# Initial chronological partitions (used only to define the temporal regions).
train_raw = df_sub[df_sub["step"] <= 520].copy()
val_raw = df_sub[(df_sub["step"] >= 521) & (df_sub["step"] <= 632)].copy()
test_raw = df_sub[(df_sub["step"] >= 633) & (df_sub["step"] <= 743)].copy()

# Recombine and stably sort before constructing historical features.
# Stable sorting preserves original row order among transactions sharing the same step.
df_all = pd.concat([train_raw, val_raw, test_raw], axis=0)
df_all = df_all.sort_values("step", kind="stable").copy()

# Time features
df_all["hour"] = df_all["step"] % 24
df_all["day"] = df_all["step"] // 24
df_all["is_night"] = (df_all["hour"] < 6).astype(int)

# Transaction type
df_all["type_transfer"] = (df_all["type"] == "TRANSFER").astype(int)

# Historical origin-account features: preceding observations only
df_all["origin_prev_count"] = df_all.groupby("nameOrig").cumcount()

origin_cumsum = (
    df_all.groupby("nameOrig")["amount"].cumsum() - df_all["amount"]
)

df_all["origin_prev_avg_amount"] = (
    origin_cumsum
    / df_all["origin_prev_count"].replace(0, np.nan)
).fillna(0)

# Historical destination-account count
df_all["dest_prev_count"] = df_all.groupby("nameDest").cumcount()

# Current amount relative to preceding origin-account average
df_all["amount_to_origin_avg"] = (
    df_all["amount"]
    / df_all["origin_prev_avg_amount"].replace(0, np.nan)
)
df_all["amount_to_origin_avg"] = (
    df_all["amount_to_origin_avg"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1.0)
)

# Recreate chronological splits after feature engineering.
train = df_all[df_all["step"] <= 520].copy()
val = df_all[(df_all["step"] >= 521) & (df_all["step"] <= 632)].copy()
test = df_all[(df_all["step"] >= 633) & (df_all["step"] <= 743)].copy()

for name, split in [("Train", train), ("Validation", val), ("Test", test)]:
    print(
        f"{name:10s}: {len(split):,} transactions | "
        f"{int(split['isFraud'].sum()):,} fraud cases | "
        f"steps {int(split['step'].min())}-{int(split['step'].max())}"
    )

assert not df_all[FEATURE_COLS].isnull().any().any()

In [ ]:
X_train = train[FEATURE_COLS].copy()
y_train = train["isFraud"].copy()

X_val = val[FEATURE_COLS].copy()
y_val = val["isFraud"].copy()

X_test = test[FEATURE_COLS].copy()
y_test = test["isFraud"].copy()

# Sampling used for RF, Clean XGBoost, and LightGBM in the original final pipeline:
# keep every fraud case and sample 20 legitimate transactions per fraud case.
train_fraud = train[train["isFraud"] == 1].copy()
train_legit = train[train["isFraud"] == 0].copy()

N_LEGIT_PER_FRAUD = 20
legit_sample = train_legit.sample(
    n=len(train_fraud) * N_LEGIT_PER_FRAUD,
    random_state=RANDOM_SEED,
)

train_sample = pd.concat([train_fraud, legit_sample], axis=0)
train_sample = train_sample.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

X_train_sample = train_sample[FEATURE_COLS].copy()
y_train_sample = train_sample["isFraud"].copy()

print("Full training shape:", X_train.shape)
print("Sampled training shape:", X_train_sample.shape)
print("Sampled fraud cases:", int(y_train_sample.sum()))
print("Sampled legitimate cases:", int((y_train_sample == 0).sum()))

## 3. Train the four models used in the final paper

In [ ]:
# Logistic Regression — full chronologically earlier training set
log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
log_reg.fit(X_train, y_train)

# Random Forest — sampled training set
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf_model.fit(X_train_sample, y_train_sample)

# Clean XGBoost — sampled training set, no class-weight adjustment
xgb_model_clean = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
xgb_model_clean.fit(X_train_sample, y_train_sample)

# LightGBM — sampled training set
lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)
lgb_model.fit(X_train_sample, y_train_sample)

VAL_PROBS = {
    "Logistic Regression": log_reg.predict_proba(X_val)[:, 1],
    "Random Forest": rf_model.predict_proba(X_val)[:, 1],
    "Clean XGBoost": xgb_model_clean.predict_proba(X_val)[:, 1],
    "LightGBM": lgb_model.predict_proba(X_val)[:, 1],
}

TEST_PROBS = {
    "Logistic Regression": log_reg.predict_proba(X_test)[:, 1],
    "Random Forest": rf_model.predict_proba(X_test)[:, 1],
    "Clean XGBoost": xgb_model_clean.predict_proba(X_test)[:, 1],
    "LightGBM": lgb_model.predict_proba(X_test)[:, 1],
}

print("All four final models trained and probabilities generated.")

## 4. Validation-only threshold selection and final held-out test evaluation

In [ ]:
def threshold_metrics(y_true, probabilities, threshold):
    pred = (probabilities >= threshold).astype(int)
    precision = precision_score(y_true, pred, zero_division=0)
    recall = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    fpr = fp / (fp + tn)
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


threshold_grid = np.arange(0.01, 1.00, 0.01)
selected_rows = []

for model_name, val_prob in VAL_PROBS.items():
    best = None
    for threshold in threshold_grid:
        metrics = threshold_metrics(y_val, val_prob, threshold)
        row = {
            "Model": model_name,
            "threshold": float(threshold),
            **metrics,
        }
        # Preserve the original notebook's behavior:
        # first (lowest) threshold wins if F1 is tied.
        if best is None or row["f1"] > best["f1"]:
            best = row
    selected_rows.append(best)

validation_threshold_results = pd.DataFrame(selected_rows)
validation_threshold_results.to_csv(
    RESULTS_DIR / "validation_threshold_results.csv",
    index=False,
)

validation_threshold_results

In [ ]:
final_rows = []

for _, row in validation_threshold_results.iterrows():
    model_name = row["Model"]
    threshold = float(row["threshold"])
    prob = TEST_PROBS[model_name]
    m = threshold_metrics(y_test, prob, threshold)

    final_rows.append({
        "Model": model_name,
        "Threshold": threshold,
        "PR-AUC": average_precision_score(y_test, prob),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "FPR": m["fpr"],
        "TN": m["TN"],
        "FP": m["FP"],
        "FN": m["FN"],
        "TP": m["TP"],
    })

final_threshold_results = pd.DataFrame(final_rows)
final_threshold_results.to_csv(
    RESULTS_DIR / "final_threshold_results.csv",
    index=False,
)

final_threshold_results

## 5. Platt calibration of Clean XGBoost

In [ ]:
xgb_clean_val_prob = VAL_PROBS["Clean XGBoost"]
xgb_clean_test_prob = TEST_PROBS["Clean XGBoost"]

brier_val_uncal = brier_score_loss(y_val, xgb_clean_val_prob)
brier_test_uncal = brier_score_loss(y_test, xgb_clean_test_prob)

platt_calibrator = LogisticRegression(
    random_state=RANDOM_SEED,
    max_iter=1000,
)
platt_calibrator.fit(
    xgb_clean_val_prob.reshape(-1, 1),
    y_val,
)

xgb_cal_val_prob = platt_calibrator.predict_proba(
    xgb_clean_val_prob.reshape(-1, 1)
)[:, 1]
xgb_cal_test_prob = platt_calibrator.predict_proba(
    xgb_clean_test_prob.reshape(-1, 1)
)[:, 1]

brier_val_cal = brier_score_loss(y_val, xgb_cal_val_prob)
brier_test_cal = brier_score_loss(y_test, xgb_cal_test_prob)

calibration_results = pd.DataFrame([
    {
        "Dataset": "Validation",
        "Uncalibrated_Brier": brier_val_uncal,
        "Calibrated_Brier": brier_val_cal,
        "Improvement_Percent": (
            (brier_val_uncal - brier_val_cal) / brier_val_uncal
        ) * 100,
    },
    {
        "Dataset": "Test",
        "Uncalibrated_Brier": brier_test_uncal,
        "Calibrated_Brier": brier_test_cal,
        "Improvement_Percent": (
            (brier_test_uncal - brier_test_cal) / brier_test_uncal
        ) * 100,
    },
])

calibration_results.to_csv(
    RESULTS_DIR / "calibration_results.csv",
    index=False,
)

calibration_results

In [ ]:
# Preserve the original calibrated validation-threshold search range.
calibrated_threshold_grid = np.arange(0.001, 0.322, 0.001)
cal_rows = []

for threshold in calibrated_threshold_grid:
    m = threshold_metrics(y_val, xgb_cal_val_prob, threshold)
    cal_rows.append({
        "threshold": float(threshold),
        **m,
    })

calibrated_threshold_search = pd.DataFrame(cal_rows)
best_idx = calibrated_threshold_search["f1"].idxmax()
best_calibrated_threshold = calibrated_threshold_search.loc[best_idx].copy()

calibrated_threshold = float(best_calibrated_threshold["threshold"])
cal_m = threshold_metrics(
    y_test,
    xgb_cal_test_prob,
    calibrated_threshold,
)

calibrated_final_results = pd.DataFrame([{
    "Model": "Platt-Calibrated Clean XGBoost",
    "Threshold": calibrated_threshold,
    "PR-AUC": average_precision_score(y_test, xgb_cal_test_prob),
    "ROC-AUC": roc_auc_score(y_test, xgb_cal_test_prob),
    "Precision": cal_m["precision"],
    "Recall": cal_m["recall"],
    "F1": cal_m["f1"],
    "FPR": cal_m["fpr"],
    "TN": cal_m["TN"],
    "FP": cal_m["FP"],
    "FN": cal_m["FN"],
    "TP": cal_m["TP"],
}])

calibrated_threshold_search.to_csv(
    RESULTS_DIR / "calibrated_threshold_search.csv",
    index=False,
)
pd.DataFrame([best_calibrated_threshold]).to_csv(
    RESULTS_DIR / "best_calibrated_threshold.csv",
    index=False,
)
calibrated_final_results.to_csv(
    RESULTS_DIR / "final_calibrated_xgboost_results.csv",
    index=False,
)

calibrated_final_results

In [ ]:
master_results = pd.concat(
    [final_threshold_results, calibrated_final_results],
    ignore_index=True,
).sort_values("F1", ascending=False).reset_index(drop=True)

master_results.to_csv(
    RESULTS_DIR / "final_master_results.csv",
    index=False,
)

master_results

## 6. Calibration and model-comparison figures

In [ ]:
# Reliability diagram
prob_true_uncal, prob_pred_uncal = calibration_curve(
    y_test,
    xgb_clean_test_prob,
    n_bins=10,
    strategy="quantile",
)
prob_true_cal, prob_pred_cal = calibration_curve(
    y_test,
    xgb_cal_test_prob,
    n_bins=10,
    strategy="quantile",
)

plt.figure(figsize=(7, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.plot(
    prob_pred_uncal,
    prob_true_uncal,
    marker="o",
    label="Uncalibrated Clean XGBoost",
)
plt.plot(
    prob_pred_cal,
    prob_true_cal,
    marker="s",
    label="Platt-Calibrated Clean XGBoost",
)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraud frequency")
plt.title("Reliability Diagram")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "reliability_diagram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Brier-score comparison
datasets = ["Validation", "Test"]
uncalibrated_scores = [brier_val_uncal, brier_test_uncal]
calibrated_scores = [brier_val_cal, brier_test_cal]

x = np.arange(len(datasets))
width = 0.35

plt.figure(figsize=(7, 6))
plt.bar(x - width / 2, uncalibrated_scores, width, label="Uncalibrated")
plt.bar(x + width / 2, calibrated_scores, width, label="Platt-Calibrated")
plt.xticks(x, datasets)
plt.ylabel("Brier Score")
plt.title("Probability Calibration Improvement")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "brier_score_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Final model comparison
models = master_results["Model"].tolist()
metrics = ["Precision", "Recall", "F1", "FPR"]
x = np.arange(len(models))
width = 0.18

plt.figure(figsize=(12, 6))
for i, metric in enumerate(metrics):
    offset = (i - 1.5) * width
    plt.bar(x + offset, master_results[metric], width, label=metric)

plt.xticks(x, models, rotation=20, ha="right")
plt.ylabel("Score")
plt.title("Final Fraud Detection Model Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "final_model_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Adaptive top-1% risk prioritization within 24-step windows

In [ ]:
adaptive_test = X_test.copy()
adaptive_test["fraud_probability"] = xgb_clean_test_prob
adaptive_test["actual_fraud"] = np.asarray(y_test)
adaptive_test["time_step"] = adaptive_test["step"].values

WINDOW_SIZE = 24
adaptive_test["time_window"] = (
    adaptive_test["time_step"] // WINDOW_SIZE
)

window_thresholds = (
    adaptive_test.groupby("time_window")["fraud_probability"]
    .quantile(0.99)
    .rename("adaptive_threshold")
    .reset_index()
)

adaptive_test = adaptive_test.merge(
    window_thresholds,
    on="time_window",
    how="left",
)

adaptive_test["adaptive_prediction"] = (
    adaptive_test["fraud_probability"]
    >= adaptive_test["adaptive_threshold"]
).astype(int)

adaptive_m = threshold_metrics(
    adaptive_test["actual_fraud"].values,
    adaptive_test["adaptive_prediction"].values.astype(float),
    0.5,
)

# threshold_metrics expects probabilities; passing 0/1 predictions with threshold 0.5
# reproduces the standard binary metrics.
adaptive_results = pd.DataFrame([{
    "Method": "Adaptive 1% Time-Window Risk Scoring",
    "Window_Size": WINDOW_SIZE,
    "Precision": adaptive_m["precision"],
    "Recall": adaptive_m["recall"],
    "F1": adaptive_m["f1"],
    "FPR": adaptive_m["fpr"],
    "TN": adaptive_m["TN"],
    "FP": adaptive_m["FP"],
    "FN": adaptive_m["FN"],
    "TP": adaptive_m["TP"],
}])

adaptive_results.to_csv(
    RESULTS_DIR / "adaptive_risk_results.csv",
    index=False,
)

fixed_clean = final_threshold_results[
    final_threshold_results["Model"] == "Clean XGBoost"
].iloc[0]

comparison = pd.DataFrame([
    {
        "Method": "Fixed Threshold Clean XGBoost",
        "Threshold / Policy": f"{fixed_clean['Threshold']:.3f}",
        "Precision": fixed_clean["Precision"],
        "Recall": fixed_clean["Recall"],
        "F1": fixed_clean["F1"],
        "FPR": fixed_clean["FPR"],
        "TN": int(fixed_clean["TN"]),
        "FP": int(fixed_clean["FP"]),
        "FN": int(fixed_clean["FN"]),
        "TP": int(fixed_clean["TP"]),
    },
    {
        "Method": "Adaptive 1% Time-Window Risk Scoring",
        "Threshold / Policy": "Top 1% per 24-step window",
        "Precision": adaptive_m["precision"],
        "Recall": adaptive_m["recall"],
        "F1": adaptive_m["f1"],
        "FPR": adaptive_m["fpr"],
        "TN": adaptive_m["TN"],
        "FP": adaptive_m["FP"],
        "FN": adaptive_m["FN"],
        "TP": adaptive_m["TP"],
    },
])

comparison.to_csv(
    RESULTS_DIR / "fixed_vs_adaptive_comparison.csv",
    index=False,
)

comparison

## 8. SHAP explainability

In [ ]:
import shap

SHAP_SAMPLE_SIZE = min(5000, len(X_test))
X_shap = X_test.sample(
    n=SHAP_SAMPLE_SIZE,
    random_state=RANDOM_SEED,
)

explainer = shap.TreeExplainer(xgb_model_clean)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_values_plot = shap_values[1]
else:
    shap_values_plot = np.asarray(shap_values)

shap_importance = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Mean_Absolute_SHAP": np.abs(shap_values_plot).mean(axis=0),
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False,
).reset_index(drop=True)

shap_importance.to_csv(
    RESULTS_DIR / "shap_feature_importance.csv",
    index=False,
)

xai_results = shap_importance.head(10).copy()
xai_results.insert(0, "Rank", range(1, len(xai_results) + 1))
xai_results.to_csv(
    RESULTS_DIR / "xai_shap_results.csv",
    index=False,
)

shap_importance

In [ ]:
# SHAP summary plot
shap.summary_plot(
    shap_values_plot,
    X_shap,
    show=False,
)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "shap_summary_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

# SHAP global feature-importance bar plot
shap.summary_plot(
    shap_values_plot,
    X_shap,
    plot_type="bar",
    show=False,
)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "shap_feature_importance_bar.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

# Local explanation for the fraud case in the SHAP sample with the highest
# Clean-XGBoost predicted fraud probability.
fraud_indices = X_shap.index[
    y_test.loc[X_shap.index].values == 1
]

if len(fraud_indices) == 0:
    raise RuntimeError("No fraud case was sampled for the local SHAP explanation.")

fraud_probs = xgb_model_clean.predict_proba(
    X_shap.loc[fraud_indices]
)[:, 1]

selected_index = fraud_indices[np.argmax(fraud_probs)]
selected_position = X_shap.index.get_loc(selected_index)

base_value = explainer.expected_value
if isinstance(base_value, (list, tuple, np.ndarray)):
    base_array = np.asarray(base_value).reshape(-1)
    base_value = base_array[-1]

explanation = shap.Explanation(
    values=shap_values_plot[selected_position],
    base_values=base_value,
    data=X_shap.loc[selected_index].values,
    feature_names=FEATURE_COLS,
)

shap.plots.waterfall(
    explanation,
    max_display=10,
    show=False,
)
plt.savefig(
    FIGURES_DIR / "shap_local_fraud_explanation.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

local_explanation_summary = pd.DataFrame([{
    "Selected_Transaction_Index": selected_index,
    "True_Label": 1,
    "Predicted_Fraud_Probability": float(fraud_probs.max()),
    "SHAP_Sample_Size": len(X_shap),
    "Fraud_Cases_In_SHAP_Sample": len(fraud_indices),
}])

local_explanation_summary.to_csv(
    RESULTS_DIR / "shap_local_explanation_metadata.csv",
    index=False,
)

local_explanation_summary

## 9. Save reproducibility metadata

In [ ]:
import sklearn
import xgboost
import lightgbm

reproducibility_config = {
    "project": {
        "name": "BIBM Fraud Detection Project",
        "task": "Fraud Detection",
        "dataset": "PaySim",
    },
    "dataset": {
        "total_transactions": int(len(df)),
        "total_fraud_cases": int(df["isFraud"].sum()),
        "overall_fraud_rate": float(df["isFraud"].mean()),
    },
    "data_subset": {
        "description": "CASH_OUT and TRANSFER transactions",
        "transactions": int(len(df_sub)),
        "fraud_cases": int(df_sub["isFraud"].sum()),
    },
    "chronological_split": {
        "train_steps": "1-520",
        "validation_steps": "521-632",
        "test_steps": "633-743",
        "train_transactions": int(len(train)),
        "validation_transactions": int(len(val)),
        "test_transactions": int(len(test)),
        "train_fraud_cases": int(train["isFraud"].sum()),
        "validation_fraud_cases": int(val["isFraud"].sum()),
        "test_fraud_cases": int(test["isFraud"].sum()),
    },
    "features": FEATURE_COLS,
    "feature_engineering": {
        "historical_features": (
            "Sequential cumulative features using preceding observations only"
        ),
        "current_transaction_excluded": True,
        "sorting": "step with stable ordering",
        "same_step_handling": (
            "Original dataset order preserved within identical steps"
        ),
    },
    "random_seed": RANDOM_SEED,
    "thresholds": {
        row["Model"]: float(row["threshold"])
        for _, row in validation_threshold_results.iterrows()
    } | {
        "Platt-Calibrated Clean XGBoost": calibrated_threshold
    },
    "calibration": {
        "method": "Platt Scaling",
        "base_model": "Clean XGBoost",
        "calibration_data": "Validation set only",
        "test_data_used_for_fitting": False,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "lightgbm": lightgbm.__version__,
        "shap": shap.__version__,
    },
}

with open(CONFIG_DIR / "reproducibility_config.json", "w") as f:
    json.dump(reproducibility_config, f, indent=4)

print("Saved:", CONFIG_DIR / "reproducibility_config.json")

## 10. Archived-result consistency check

In [ ]:
# Values reported in the accepted-paper project archive.
# This cell does not force the results; it only checks the fresh run.

EXPECTED = {
    "clean_xgb_f1": 0.5369332654100866,
    "clean_xgb_pr_auc": 0.5658165276462193,
    "clean_xgb_roc_auc": 0.9068394249322168,
    "test_brier_uncal": 0.11408600810469051,
    "test_brier_cal": 0.024513744220956812,
    "adaptive_fp": 117,
    "adaptive_recall": 0.21221864951768488,
}

clean_row = final_threshold_results[
    final_threshold_results["Model"] == "Clean XGBoost"
].iloc[0]

observed = {
    "clean_xgb_f1": float(clean_row["F1"]),
    "clean_xgb_pr_auc": float(clean_row["PR-AUC"]),
    "clean_xgb_roc_auc": float(clean_row["ROC-AUC"]),
    "test_brier_uncal": float(brier_test_uncal),
    "test_brier_cal": float(brier_test_cal),
    "adaptive_fp": int(adaptive_results.loc[0, "FP"]),
    "adaptive_recall": float(adaptive_results.loc[0, "Recall"]),
}

for key, expected_value in EXPECTED.items():
    observed_value = observed[key]
    if isinstance(expected_value, int):
        ok = observed_value == expected_value
    else:
        ok = np.isclose(
            observed_value,
            expected_value,
            rtol=1e-5,
            atol=1e-7,
        )
    print(
        f"{key:22s} observed={observed_value!r} "
        f"expected={expected_value!r} "
        f"{'OK' if ok else 'CHECK'}"
    )

## Interpretation limits

This notebook reproduces the experimental pipeline used for the accepted BIBM paper, but the evidence remains limited to the **synthetic PaySim benchmark**. The reported precision, recall, thresholds, SHAP rankings, and adaptive-policy results should not be interpreted as direct estimates of fraud patterns or deployed-model performance in Bangladesh.

Operational use would require governed local transaction data, privacy/regulatory review, feature-availability checks, drift monitoring, cost-sensitive threshold/policy selection, and human oversight.